In [1]:
import os, json, math, time, ast
from typing import Dict, Any, List, Tuple
import pandas as pd
from dotenv import load_dotenv
from jsonschema import validate
from openai import OpenAI

In [2]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"     # 필요시 "gpt-4o" 등으로 변경
TEMPERATURE = 0.6
TOP_P = 0.9
MAX_TOKENS = 1400
MAX_RETRIES = 3           # LLM 재시도 (형식 실패 시)
MAX_GENERATION_TRIES = 6  # 페르소나 1명을 '중복 피해서' 뽑기 위한 최대 시도
N_PERSONAS = 20

INPUT_CSV = "data/sample_submission.csv"
OUT_PERSONAS_JSONL = "data/personas_20.jsonl"
OUT_FORECAST_CSV = "data/forecast_by_sku_month.csv"

In [3]:
persona_required = [
    "name","age_group","gender","family_structure","job","income_level",
    "education_level","region","household_size","budget_food_month_krw",
    "diet_preference","lifestyle","likes_food","health_constraints","main_channels",
    "digital_literacy","price_sensitivity","brand_loyalty","eco_friendliness",
    "flavor_preferences","summary_tag"
]

schema = {
    "type": "object",
    "required": ["persona", "purchase_model", "assumptions", "population_weight"],
    "properties": {
        "persona": {
            "type": "object",
            "required": persona_required,
            "properties": {
                "name": {"type":"string"},
                "age_group": {"type":"string"},
                "gender": {"type":"string"},
                "family_structure": {"type":"string"},
                "job": {"type":"string"},
                "income_level": {"type":"string"},
                "education_level": {"type":"string"},
                "region": {"type":"string"},
                "household_size": {"type":"integer","minimum":1},
                "budget_food_month_krw": {"type":"number","minimum":0},
                "diet_preference": {"type":"string"},
                "lifestyle": {"type":"array","items":{"type":"string"},"minItems":3},
                "likes_food": {"type":"array","items":{"type":"string"},"minItems":3},
                "health_constraints": {"type":"array","items":{"type":"string"}},
                "main_channels": {"type":"array","items":{"type":"string"},"minItems":1},
                "digital_literacy": {"type":"string"},
                "price_sensitivity": {"type":"number","minimum":0,"maximum":1},
                "brand_loyalty": {"type":"number","minimum":0,"maximum":1},
                "eco_friendliness": {"type":"number","minimum":0,"maximum":1},
                "flavor_preferences": {
                    "type":"object",
                    "required":["spicy_tolerance","sweet_preference","savory_preference","sour_preference","umami_preference"],
                    "properties":{
                        "spicy_tolerance":{"type":"number","minimum":0,"maximum":1},
                        "sweet_preference":{"type":"number","minimum":0,"maximum":1},
                        "savory_preference":{"type":"number","minimum":0,"maximum":1},
                        "sour_preference":{"type":"number","minimum":0,"maximum":1},
                        "umami_preference":{"type":"number","minimum":0,"maximum":1}
                    },
                    "additionalProperties": False
                },
                "summary_tag": {"type":"string"}
            },
            "additionalProperties": False
        },
        "product_fit": {
            "type":"object",
            "properties":{
                "need_state":{"type":"array","items":{"type":"string"}},
                "key_triggers":{"type":"array","items":{"type":"string"}},
                "barriers":{"type":"array","items":{"type":"string"}}
            },
            "additionalProperties": True
        },
        "purchase_model": {
            "type":"object",
            "required":[
                "buy_probability","buy_probability_ci95",
                "avg_units_per_purchase","avg_units_ci95",
                "monthly_purchase_frequency","seasonality_notes","promo_uplift_factor"
            ],
            "properties":{
                "buy_probability":{"type":"number","minimum":0,"maximum":1},
                "buy_probability_ci95":{"type":"array","items":{"type":"number"},"minItems":2,"maxItems":2},
                "avg_units_per_purchase":{"type":"number","minimum":0},
                "avg_units_ci95":{"type":"array","items":{"type":"number"},"minItems":2,"maxItems":2},
                "monthly_purchase_frequency":{"type":"array","items":{"type":"number","minimum":0},"minItems":12,"maxItems":12},
                "seasonality_notes":{"type":"array","items":{"type":"string"}},
                "promo_uplift_factor":{"type":"number","minimum":0.5,"maximum":2.0}
            },
            "additionalProperties": False
        },
        "assumptions": {  # LLM 자체 추정
            "type":"object",
            "required":["inferred_base_conversion","inferred_base_monthly_freq","seasonality_weights_mean1"],
            "properties":{
                "inferred_base_conversion":{"type":"number","minimum":0,"maximum":1},
                "inferred_base_monthly_freq":{"type":"number","minimum":0},
                "seasonality_weights_mean1":{
                    "type":"array","items":{"type":"number","minimum":0},"minItems":12,"maxItems":12
                },
                "notes":{"type":"string"}
            },
            "additionalProperties": True
        },
        "population_weight": {"type":"number","minimum":0,"maximum":1}
    },
    "additionalProperties": False
}

In [4]:
def normalize_to_mean_one(weights, ndigits=3):
    """길이 12 배열을 평균 1.000로 스케일링하고, 반올림 오차는 마지막 원소로 보정."""
    if not weights or len(weights) != 12:
        raise ValueError("seasonality_weights_mean1 길이 12 아님")
    w = [max(0.0, float(x)) for x in weights]  # 음수 방지
    m = sum(w) / 12.0
    if m == 0:
        w = [1.0]*12
    else:
        w = [x / m for x in w]
    w = [round(x, ndigits) for x in w]
    diff = round(12.0 - sum(w), ndigits)
    w[-1] = round(w[-1] + diff, ndigits)
    return w

In [5]:
def build_prompt(product: Dict[str, Any], distinct_notes: str = "") -> str:
    # distinct_notes: 이미 선택된 페르소나 요약/금지 조건(중복회피)을 포함
    return f"""
당신은 한국 식품시장 전문가이자 데이터 정합성 검수자다.
아래 "입력"만으로 이 제품의 잠재 핵심 고객을 대표하는
"현실적인 소비자 페르소나 1명"과 그 페르소나의 구매행동을 산출하라.

중요:
- 어떠한 사전 앵커(base_conversion, base_monthly_freq, seasonal_notes)도 주어지지 않는다.
- 카테고리/가격/패키지/채널/맛 프로파일을 근거로
  '시장 평균 전환율', '월 평균 구매빈도', '월별 시즌 가중치(평균=1.000)'를 스스로 추정하라.
- 추정치는 "assumptions"에 기입하라. seasonality_weights_mean1의 합은 12.000이어야 한다(평균 1.000).
  소수 셋째 자리 반올림 후 총합이 12.000이 아니면 마지막 원소를 미세 조정하라.
- 출력은 반드시 "출력 스키마(JSON)"만 반환. 다른 문장 금지.
- 페르소나 속성은 최소 10개 이상 채워라. 내부 모순 금지.
- 확률은 0~1, 월별 빈도는 길이 12 비음수 실수, 구매당 평균 수량은 양의 실수.
- 모든 실수는 소수 3자리 이내. 95% 신뢰구간 병기.
- 다음과 같이 기존 페르소나와 겹치지 않도록 하라:
  {distinct_notes}

[필수 페르소나 필드]
{json.dumps(persona_required, ensure_ascii=False)}

[출력 스키마(JSON)]
{{
  "persona": {{
    "name": "<한글 가명>",
    "age_group": "<20대|30대|40대|50대|60대 이상>",
    "gender": "<남|여>",
    "family_structure": "<1인 가구|부모동거|부부|자녀1명|자녀2명 이상|기타>",
    "job": "<직업>",
    "income_level": "<소득구간>",
    "education_level": "<최종학력>",
    "region": "<거주권역/도시>",
    "household_size": <정수>,
    "budget_food_month_krw": <월 식비 예산(원)>,
    "diet_preference": "<일반|저염|고단백|채식 등>",
    "lifestyle": ["3~6개"],
    "likes_food": ["3~6개"],
    "health_constraints": ["0~3개"],
    "main_channels": ["1~3개 (예: 자사몰, 쿠팡, 대형마트, 편의점)"],
    "digital_literacy": "<낮음|보통|높음>",
    "price_sensitivity": <0~1>,
    "brand_loyalty": <0~1>,
    "eco_friendliness": <0~1>,
    "flavor_preferences": {{
      "spicy_tolerance": <0~1>,
      "sweet_preference": <0~1>,
      "savory_preference": <0~1>,
      "sour_preference": <0~1>,
      "umami_preference": <0~1>
    }},
    "summary_tag": "<핵심 요약>"
  }},
  "product_fit": {{
    "need_state": ["2~4개"],
    "key_triggers": ["2~4개"],
    "barriers": ["1~3개"]
  }},
  "purchase_model": {{
    "buy_probability": <0~1>,
    "buy_probability_ci95": [<하한>, <상한>],
    "avg_units_per_purchase": <양수>,
    "avg_units_ci95": [<하한>, <상한>],
    "monthly_purchase_frequency": [m1, m2, ..., m12],
    "seasonality_notes": ["월별 패턴 이유 1~2줄"],
    "promo_uplift_factor": <0.5~2.0>
  }},
  "assumptions": {{
    "inferred_base_conversion": <0~1>,
    "inferred_base_monthly_freq": <0 이상>,
    "seasonality_weights_mean1": [w1, w2, ..., w12],  // 평균=1.000(합=12.000)
    "notes": "<추정 근거 한 줄>"
  }},
  "population_weight": 1.0
}}

[입력]
{json.dumps(product, ensure_ascii=False, indent=2)}
""".strip()

In [6]:
def call_llm_once(prompt: str) -> str:
    rsp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":"당신은 데이터 분석가이자 소비자 리서치 전문가입니다. JSON만 출력하세요."},
            {"role":"user","content": prompt}
        ],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS
    )
    return rsp.choices[0].message.content

def get_persona_json(prompt: str) -> Dict[str, Any]:
    last_err = None
    for attempt in range(1, MAX_RETRIES+1):
        raw = call_llm_once(prompt).strip()
        if raw.startswith("```"):
            raw = raw.strip("`").replace("json\n","").replace("json\r\n","")
        try:
            data = json.loads(raw)

            # 시즌 가중치 자동 정규화 (검증 전 수행)
            seas = data.get("assumptions", {}).get("seasonality_weights_mean1", None)
            if not seas or len(seas) != 12:
                raise ValueError("seasonality_weights_mean1 길이 12 아님")
            seas_norm = normalize_to_mean_one(seas, ndigits=3)
            data["assumptions"]["seasonality_weights_mean1"] = seas_norm

            # 스키마 검증
            validate(instance=data, schema=schema)

            # 페르소나 속성 10개 이상 채움 확인
            persona = data.get("persona", {})
            filled = [k for k,v in persona.items() if v not in (None,"",[],{})]
            if len(filled) < 10:
                raise ValueError("페르소나 속성 10개 이상 미충족")

            # 월빈도 길이 검사
            mpf = data["purchase_model"]["monthly_purchase_frequency"]
            if len(mpf) != 12:
                raise ValueError("monthly_purchase_frequency 길이 12 아님")

            return data
        except Exception as e:
            last_err = e
            time.sleep(0.7 + 0.3*attempt)
    raise RuntimeError(f"LLM JSON 검증/파싱 실패: {last_err}")

In [7]:
def cosine(v1: List[float], v2: List[float]) -> float:
    a = math.sqrt(sum(x*x for x in v1)) or 1e-9
    b = math.sqrt(sum(x*x for x in v2)) or 1e-9
    return sum(x*y for x,y in zip(v1,v2)) / (a*b)

def jaccard(set1: set, set2: set) -> float:
    if not set1 and not set2: return 1.0
    return len(set1 & set2) / float(len(set1 | set2) or 1)

def extract_signature(p: Dict[str,Any]) -> Dict[str,Any]:
    """페르소나 '서명' 추출: 중복 판정에 쓰는 핵심 축들"""
    per = p["persona"]
    fp = per["flavor_preferences"]
    vec = [
        float(fp.get("spicy_tolerance",0)),
        float(fp.get("sweet_preference",0)),
        float(fp.get("savory_preference",0)),
        float(fp.get("sour_preference",0)),
        float(fp.get("umami_preference",0)),
        float(per.get("price_sensitivity",0)),
        float(per.get("brand_loyalty",0)),
        float(per.get("eco_friendliness",0)),
    ]
    cats = {
        "age_group": per.get("age_group",""),
        "gender": per.get("gender",""),
        "family_structure": per.get("family_structure",""),
        "diet_preference": per.get("diet_preference",""),
        "region": per.get("region",""),
    }
    sets = {
        "lifestyle": set(map(str, per.get("lifestyle",[]))),
        "likes_food": set(map(str, per.get("likes_food",[]))),
        "channels": set(map(str, per.get("main_channels",[]))),
    }
    return {"vec": vec, "cats": cats, "sets": sets}

def is_distinct(sig_new: Dict[str,Any], sig_old: Dict[str,Any],
                vec_thr: float=0.92, # 코사인 유사도 상한 (이상이면 너무 비슷)
                j_thr: float=0.65    # 자카드 유사도 상한
               ) -> bool:
    sim_vec = cosine(sig_new["vec"], sig_old["vec"])
    if sim_vec >= vec_thr:
        return False
    # 주요 카테고리 중 최소 한두 개는 달라야 함
    cat_equal_count = sum(sig_new["cats"].get(k)==sig_old["cats"].get(k) for k in sig_new["cats"].keys())
    if cat_equal_count >= 4:  # 5개 중 4개 이상 동일하면 과도하게 유사
        return False
    # 라이프스타일/채널/음식 자카드 유사도도 과하면 탈락
    if jaccard(sig_new["sets"]["lifestyle"], sig_old["sets"]["lifestyle"]) >= j_thr:
        return False
    if jaccard(sig_new["sets"]["channels"], sig_old["sets"]["channels"]) >= j_thr:
        return False
    return True

def build_distinct_note(existing_personas: List[Dict[str,Any]], max_examples:int=6) -> str:
    """LLM에 '이들과 겹치지 말라' 힌트를 주기 위한 요약 문자열"""
    notes = []
    for p in existing_personas[:max_examples]:
        per = p["persona"]
        fp = per["flavor_preferences"]
        notes.append(
            f"- 금지 유사 패턴: {per.get('age_group')}/{per.get('gender')}/"
            f"{per.get('family_structure')}/{per.get('region')}/채널:{','.join(per.get('main_channels',[]))}/"
            f"맛벡터(spicy:{fp.get('spicy_tolerance')}, sweet:{fp.get('sweet_preference')}, "
            f"sav:{fp.get('savory_preference')}, sour:{fp.get('sour_preference')}, umami:{fp.get('umami_preference')})"
        )
    if not notes:
        notes = ["(현재 금지 패턴 없음)"]
    return "\n  ".join(notes)


In [8]:
def flavor_match(persona_pref: Dict[str,float], sku_profile: Dict[str,float]) -> float:
    keys = ["spicy","sweet","savory","sour","umami"]
    p = [
        float(persona_pref.get("spicy_tolerance",0)),
        float(persona_pref.get("sweet_preference",0)),
        float(persona_pref.get("savory_preference",0)),
        float(persona_pref.get("sour_preference",0)),
        float(persona_pref.get("umami_preference",0)),
    ]
    s = [float(sku_profile.get(k,0)) for k in keys]
    dot = sum(a*b for a,b in zip(p,s))
    na = math.sqrt(sum(a*a for a in p)) or 1e-9
    nb = math.sqrt(sum(b*b for b in s)) or 1e-9
    score = dot/(na*nb)
    return max(0.0, min(1.0, score))

In [9]:
def generate_distinct_personas(product_for_prompt: Dict[str,Any], n:int=20) -> List[Dict[str,Any]]:
    personas: List[Dict[str,Any]] = []
    signatures: List[Dict[str,Any]] = []

    while len(personas) < n:
        distinct_notes = build_distinct_note(personas)
        prompt = build_prompt(product_for_prompt, distinct_notes=distinct_notes)

        accepted = None
        for _ in range(MAX_GENERATION_TRIES):
            cand = get_persona_json(prompt)
            sig_new = extract_signature(cand)
            ok = True
            for sig_old in signatures:
                if not is_distinct(sig_new, sig_old):
                    ok = False
                    break
            if ok:
                accepted = cand
                break
            time.sleep(0.4)  # 살짝 대기 후 재생성

        if accepted is None:
            # 기준 완화 한 번: 코사인/자카드 임계 소폭 강화
            def is_distinct_relaxed(sig_new, sig_old):
                return is_distinct(sig_new, sig_old, vec_thr=0.95, j_thr=0.75)
            for _ in range(MAX_GENERATION_TRIES):
                cand = get_persona_json(prompt)
                sig_new = extract_signature(cand)
                ok = True
                for sig_old in signatures:
                    if not is_distinct_relaxed(sig_new, sig_old):
                        ok = False
                        break
                if ok:
                    accepted = cand
                    break

        if accepted is None:
            raise RuntimeError("충분히 겹치지 않는 페르소나 생성에 실패했습니다. (생성 로직/프롬프트 기준을 조정하세요)")
        personas.append(accepted)
        signatures.append(extract_signature(accepted))

    # population_weight 합=1 정규화
    total_w = sum(float(p.get("population_weight",1.0)) for p in personas) or 1.0
    for p in personas:
        p["population_weight"] = float(p.get("population_weight",1.0)) / total_w
    return personas

In [10]:
def monthly_forecast_for_persona(persona_json: Dict[str,Any], product: Dict[str,Any]) -> pd.DataFrame:
    pm = persona_json["purchase_model"]
    w = float(persona_json.get("population_weight", 1.0))
    price = float(product["price"])
    p = float(pm["buy_probability"])
    u = float(pm["avg_units_per_purchase"])
    freq = [float(x) for x in pm["monthly_purchase_frequency"]]
    uplift = float(pm["promo_uplift_factor"])
    flavor_factor = flavor_match(persona_json["persona"]["flavor_preferences"], product["flavor_profile"])
    units = [w * p * f * u * uplift * flavor_factor for f in freq]
    revenue = [x * price for x in units]
    return pd.DataFrame({"month": range(1,13), "expected_units": units, "expected_revenue": revenue})

def forecast_for_sku(product_row: Dict[str,Any], personas: List[Dict[str,Any]]) -> pd.DataFrame:
    # 각 페르소나별 계산 후 합산
    monthly_list = []
    for i, per in enumerate(personas):
        df_i = monthly_forecast_for_persona(per, product_row)
        df_i["persona_idx"] = i
        monthly_list.append(df_i)
    by_persona = pd.concat(monthly_list, ignore_index=True)
    agg = by_persona.groupby("month", as_index=False).agg({"expected_units":"sum","expected_revenue":"sum"})
    return agg

In [11]:
def parse_listish(val) -> List[str]:
    if pd.isna(val): return []
    if isinstance(val, list): return [str(x).strip() for x in val]
    s = str(val).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = ast.literal_eval(s)
            return [str(x).strip() for x in arr]
        except Exception:
            pass
    # 쉼표/슬래시 구분 허용
    return [x.strip() for x in s.replace("/",",").split(",") if x.strip()]

def parse_flavor_profile(val) -> Dict[str,float]:
    # JSON 또는 dict string 우선 파싱
    default = {"spicy":0.2,"sweet":0.2,"savory":0.5,"sour":0.1,"umami":0.6}
    if pd.isna(val): return default
    if isinstance(val, dict): return {k: float(val.get(k, default.get(k,0))) for k in default}
    s = str(val).strip()
    try:
        d = json.loads(s)
        return {k: float(d.get(k, default.get(k,0))) for k in default}
    except Exception:
        try:
            d = ast.literal_eval(s)
            if isinstance(d, dict):
                return {k: float(d.get(k, default.get(k,0))) for k in default}
        except Exception:
            pass
    # 키워드 힌트로 조정
    s_lower = s.lower()
    fp = default.copy()
    if "매운" in s or "spicy" in s_lower: fp["spicy"] = max(fp["spicy"], 0.6)
    if "달콤" in s or "sweet" in s_lower: fp["sweet"] = max(fp["sweet"], 0.5)
    if "고소" in s or "풍미" in s or "savory" in s_lower: fp["savory"] = max(fp["savory"], 0.6)
    if "감칠맛" in s or "umami" in s_lower: fp["umami"] = max(fp["umami"], 0.7)
    if "새콤" in s or "sour" in s_lower: fp["sour"] = max(fp["sour"], 0.4)
    return fp

def row_to_product(row: pd.Series) -> Dict[str,Any]:
    return {
        "product_name": row.get("product_name", row.get("name","상품")),
        "category": row.get("category","식품"),
        "price": float(row.get("price", 10000.0)),
        "pack_info": row.get("pack_info", ""),
        "key_attributes": parse_listish(row.get("key_attributes","")),
        "channels": parse_listish(row.get("channels","자사몰,쿠팡")),
        "flavor_profile": parse_flavor_profile(row.get("flavor_profile",""))
    }

In [12]:
def main():
    # 1) 페르소나 20명(제품 독립적 '시장 대표' 성격으로 생성하기 위해,
    #    대표 SKU 하나를 프롬프트용으로 사용)
    #    → 실무에서는 카테고리 대표 SKU(중간 가격/보편 채널)를 넣는 걸 권장
    representative_product = {
        "product_name": "카테고리 대표 SKU",
        "category": "식품",
        "price": 15000,
        "pack_info": "기본",
        "key_attributes": ["일반적인 속성"],
        "channels": ["자사몰","쿠팡","대형마트","편의점"],
        "flavor_profile": {"spicy":0.4,"sweet":0.3,"savory":0.6,"sour":0.2,"umami":0.6}
    }
    personas = generate_distinct_personas(representative_product, n=N_PERSONAS)

    # 2) 페르소나 저장(JSONL)
    with open(OUT_PERSONAS_JSONL, "w", encoding="utf-8") as f:
        for p in personas:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    # 3) SKU CSV 로드
    df = pd.read_csv(INPUT_CSV)
    # 예상되는 컬럼 힌트:
    # product_name, category, price, pack_info, key_attributes, channels, flavor_profile
    # 일부 없음 → 기본값/추론

    # 4) 각 SKU 예측 후 합치기
    out_rows = []
    for idx, row in df.iterrows():
        prod = row_to_product(row)
        agg = forecast_for_sku(prod, personas)  # month, expected_units, expected_revenue
        agg["product_name"] = prod["product_name"]
        agg["category"] = prod["category"]
        agg["price"] = prod["price"]
        # 원본 CSV에 id 컬럼 있으면 유지
        if "id" in row.index:
            agg["id"] = row["id"]
        out_rows.append(agg)

    out_df = pd.concat(out_rows, ignore_index=True)
    # 컬럼 순서 정리
    cols = ["product_name","category","price","month","expected_units","expected_revenue"]
    if "id" in out_df.columns:
        cols = ["id"] + cols
    out_df = out_df[cols]

    # 5) 저장
    out_df.to_csv(OUT_FORECAST_CSV, index=False, encoding="utf-8-sig")

    print(f"✅ 페르소나 20명 저장: {OUT_PERSONAS_JSONL}")
    print(f"✅ SKU×월 예측 저장: {OUT_FORECAST_CSV}")
    print("샘플 출력 미리보기:")
    print(out_df.head(12).to_string(index=False))

if __name__ == "__main__":
    main()

InternalServerError: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>cloudflare</center>
</body>
</html>